In [61]:
import os
import sqlite3
import sys

import pandas as pd
import zeep
from zeep import Settings
from zeep.helpers import serialize_object
from zeep.wsse.username import UsernameToken

import functools

In [50]:
username = "9c2fc57f048fd5c2f740dcf8e0a6f69c677c424289bda1736196674"
password = "d67f1b3e02ff57fb6d90fc8a770db00b"
SERVICE_WSDL_URL = "https://webservices.fd.chargepoint.com/cp_api_5.1.wsdl"


In [51]:
from zeep import Client
from zeep.wsse.username import UsernameToken

def _get_response_helper(client, api_entry_name: str, data_section_name: str, query_para_dict: dict = {}, start_record: int = 1, responses: list = None,
                         start_record_key: str = "startRecord", record_number_key: str = "recordNumber", left_iteration = 10, **kwargs):
    if responses is None:
        responses = []
    if not responses:  # is None or responses == [] 
        more_flag = 1
    else:
        # print(f"===== {responses = }")
        # print(f"======== {responses[-1].keys() = }")
        more_flag_key = list(responses[-1].keys())[-1] # Note: assume "MoreFlag" or "moreFlag" key is the last key
        if str.lower(more_flag_key) == "moreflag":
            more_flag = responses[-1][more_flag_key]
        else:
            more_flag = 0

    if more_flag == 1 and left_iteration > 0:
        method = getattr(client.service, api_entry_name)  # e.g., method = client.service.getChargingSessionData
        query_para_dict.update({start_record_key: start_record})
        response = method(query_para_dict)  # e.g., response = client.service.getChargingSessionData(kwargs)
        res_dict: dict = serialize_object(response)
        responses += [res_dict]
        new_start_record = start_record + len(response[data_section_name])
        # print(f"==={new_start_record = }")
        return _get_response_helper(client, api_entry_name, data_section_name, query_para_dict, new_start_record, responses,
                                    start_record_key, record_number_key, left_iteration-1, **kwargs)
    else:
        return responses

def get_response(client, api_entry_name: str, data_section_name: str, query_para_dict: dict = {}, start_record: int = 1,
                 start_record_key: str = "startRecord", record_number_key: str = "recordNumber", max_interation=10, **kwargs):
    return _get_response_helper(client, api_entry_name, data_section_name, query_para_dict, start_record, None,
                                start_record_key, record_number_key, **kwargs)

# SERVICE_WSDL_URL = "your_wsdl_url"  # Replace with your actual WSDL URL
# username = "your_username"  # Replace with your actual username
# password = "your_password"  # Replace with your actual password

# Initialize client with security settings
client = Client(
    SERVICE_WSDL_URL,
    wsse=UsernameToken(username, password),
)

# Example usage
start_record = 1
responses = get_response(client=client, api_entry_name="getChargingSessionData", 
                         data_section_name="ChargingSessionData", query_para_dict={})

print(f"{len(responses) = }")   
# print(responses)


# bonus: clean up responses

def construct_data_session(responses: list[dict], data_section_name: str) -> list[dict]:
    responses_data = []
    for res in responses: 
        # res_dict = serialize_object(res)
        data = res[data_section_name]
        responses_data += (data)
        return responses_data
    
responses_data = construct_data_session(responses, "ChargingSessionData")


df_responses = pd.DataFrame(responses_data)
df_responses.info()

len(responses) = 5
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype                                                        
---  ------                  --------------  -----                                                        
 0   stationID               500 non-null    object                                                       
 1   stationName             500 non-null    object                                                       
 2   portNumber              500 non-null    object                                                       
 3   Address                 500 non-null    object                                                       
 4   City                    500 non-null    object                                                       
 5   State                   500 non-null    object                                                       
 6   Country        

In [58]:
def _getChargingSessionData(kwargs):
    response = client.service.getChargingSessionData(kwargs)
    return response

def _getChargingSessionDataAll(stationID = None, sessionID = None, userID = None, stationName = None, Address = None, City = None, State = None, Country = None, postalCode = None, Proximity = None, proximityUnit = None, fromTimeStamp = None, toTimeStamp = None, startRecord = None, Geo = None, stationIDs = None, activeSessionsOnly = None):
    # searchQuery = client.get_type("ns0:sessionSearchdata")()
    searchQuery = {
    'stationID': stationID,
    'sessionID': sessionID,
    'userID': userID,
    'stationName': stationName,
    'Address': Address,
    'City': City,
    'State': State,
    'Country': Country,
    'postalCode': postalCode,
    'Proximity': Proximity,
    'proximityUnit': proximityUnit,
    'fromTimeStamp': fromTimeStamp,
    'toTimeStamp': toTimeStamp,
    'startRecord': startRecord,
    'Geo': Geo,
    'stationIDs': stationIDs,
    'activeSessionsOnly': activeSessionsOnly
}
    response = _getChargingSessionData(searchQuery)
    if response["responseCode"] != "100":
        raise Exception(f"Error: {response = }")
    return serialize_object(response)

_getChargingSessionDataAll()

OrderedDict([('responseCode', '100'),
             ('responseText', 'API input request executed successfully.'),
             ('ChargingSessionData',
              [OrderedDict([('stationID', '5:13090571'),
                            ('stationName', 'PNNL / EMSL 9'),
                            ('portNumber', '1'),
                            ('Address',
                             '650 Horn Rapids Rd, Richland, Washington, 99354, United States'),
                            ('City', 'Richland'),
                            ('State', 'Washington'),
                            ('Country', 'United States'),
                            ('postalCode', '99354'),
                            ('sessionID', 79776929),
                            ('Energy', 0.0),
                            ('startTime',
                             datetime.datetime(2024, 5, 4, 17, 27, 31, tzinfo=<isodate.tzinfo.Utc object at 0x7f0857263a90>)),
                            ('endTime',
                         

In [107]:
import functools

def paginated_api_call(api_entry_name, data_section_name, start_record_key="startRecord", more_flag_key="MoreFlag", max_iterations=10):
    def decorator_api_call(func):
        @functools.wraps(func)
        def wrapper(client, *args, **kwargs):
            responses = []
            more_flag = True
            start_record = 1
            iterations = 0

            while more_flag and iterations < max_iterations:
                # Add or update the start record in the kwargs before calling the function
                kwargs[start_record_key] = start_record

                # Get the response from the API via the wrapped function
                # response = func(client, *args, **kwargs)
                response = func(client, **kwargs)

                if response["responseCode"] != "100":
                    raise Exception(f"API error with response: {response}")

                response = serialize_object(response)
                # Process the response data
                data = response.get(data_section_name, [])
                responses.extend(data)
                
                # Check if more data is available
                more_flag = response.get(more_flag_key, "NotExist") == 1
                print(f"========{more_flag = }, {response.get(more_flag_key, 'NotExist') = }")
                if more_flag:
                    start_record += len(data)

                iterations += 1
                if iterations >= max_iterations:
                    raise Exception(f"Warning: Maximum iterations reached ({max_iterations})")

            return responses
        return wrapper
    return decorator_api_call

# This should be a wrapper around the API call you actually want to make.
def _getChargingSessionData(client, kwargs):
    return client.service.getChargingSessionData(kwargs)

# Wrap the actual data-fetching function
@paginated_api_call(api_entry_name="_getChargingSessionData", data_section_name="ChargingSessionData")
def _getChargingSessionDataAll(client, **kwargs):
    # Pass the client and parameters to the `_getChargingSessionData`
    return _getChargingSessionData(client, kwargs)


def getChargingSessionDataAPI(client, stationID: str = None, sessionID: int = None, userID = None, 
                              stationName: str = None, Address: str = None, City: str = None, State: str = None, Country = None, 
                              postalCode = None, Proximity = None, proximityUnit = None, 
                              fromTimeStamp: str = None, toTimeStamp: str = None, 
                              startRecord: int = None, Geo = None, stationIDs: list[str] = None, activeSessionsOnly = None):
    """
    Example fromTimeStamp: 2024-12-01T00:00:00, 2024-12-01
    """
    if fromTimeStamp is not None:
        fromTimeStamp = pd.to_datetime(fromTimeStamp).strftime("%Y-%m-%dT%H:%M:%S")  
    if toTimeStamp is not None:
        toTimeStamp = pd.to_datetime(toTimeStamp).strftime("%Y-%m-%dT%H:%M:%S")
    # searchQuery = client.get_type("ns0:sessionSearchdata")()
    searchQuery = {
    'stationID': stationID,
    'sessionID': sessionID,
    'userID': userID,
    'stationName': stationName,
    'Address': Address,
    'City': City,
    'State': State,
    'Country': Country,
    'postalCode': postalCode,
    'Proximity': Proximity,
    'proximityUnit': proximityUnit,
    'fromTimeStamp': fromTimeStamp,
    'toTimeStamp': toTimeStamp,
    'startRecord': startRecord,
    'Geo': Geo,
    'stationIDs': stationIDs,
    'activeSessionsOnly': activeSessionsOnly
    }
    print(f"{searchQuery = }")
    return _getChargingSessionDataAll(client, **searchQuery)
    
    
getChargingSessionDataAPI(client, sessionID=100412719)
# getChargingSessionData(client, sessionID=100412719)

searchQuery = {'stationID': None, 'sessionID': 100412719, 'userID': None, 'stationName': None, 'Address': None, 'City': None, 'State': None, 'Country': None, 'postalCode': None, 'Proximity': None, 'proximityUnit': None, 'fromTimeStamp': None, 'toTimeStamp': None, 'startRecord': None, 'Geo': None, 'stationIDs': None, 'activeSessionsOnly': None}
========more_flag = False, response.get(more_flag_key, 'NotExist') = 0


[OrderedDict([('stationID', '5:15504761'),
              ('stationName', 'PNNL / MSL5'),
              ('portNumber', '1'),
              ('Address',
               '1529 W Sequim Bay Rd, Sequim, Washington, 98382, United States'),
              ('City', 'Sequim'),
              ('State', 'Washington'),
              ('Country', 'United States'),
              ('postalCode', '98382'),
              ('sessionID', 100412719),
              ('Energy', 2.988125),
              ('startTime',
               datetime.datetime(2025, 1, 27, 22, 38, 52, tzinfo=<isodate.tzinfo.Utc object at 0x7f0857263a90>)),
              ('endTime',
               datetime.datetime(2025, 1, 28, 22, 27, 53, tzinfo=<isodate.tzinfo.Utc object at 0x7f0857263a90>)),
              ('totalChargingDuration', '00:34:57'),
              ('totalSessionDuration', '23:49:00'),
              ('userID', None),
              ('startBatteryPercentage', 0.0),
              ('stopBatteryPercentage', 0.0),
              ('recordNu

In [108]:
res = getChargingSessionDataAPI(client, fromTimeStamp="2024-12-01T00:00:00")
df_res = pd.DataFrame(res)
df_res.info()

searchQuery = {'stationID': None, 'sessionID': None, 'userID': None, 'stationName': None, 'Address': None, 'City': None, 'State': None, 'Country': None, 'postalCode': None, 'Proximity': None, 'proximityUnit': None, 'fromTimeStamp': '2024-12-01T00:00:00', 'toTimeStamp': None, 'startRecord': None, 'Geo': None, 'stationIDs': None, 'activeSessionsOnly': None}
========more_flag = True, response.get(more_flag_key, 'NotExist') = 1
========more_flag = False, response.get(more_flag_key, 'NotExist') = 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype                                                        
---  ------                  --------------  -----                                                        
 0   stationID               623 non-null    object                                                       
 1   stationName             623 non-null    object                        

In [102]:
fromTimeStamp = "2024-12-01T00:00:00"

str(pd.to_datetime(fromTimeStamp))

'2024-12-01 00:00:00'

In [ ]:
getChargingSessionDataAPI(client, fromTimeStamp="2024-12-01T00:00:00")

In [83]:
# _getChargingSessionData(client, {"fromTimeStamp":"2024-12-01T00:00:00"})
_getChargingSessionData(client, {"fromTimeStamp":"2024-12-01"})

{
    'responseCode': '100',
    'responseText': 'API input request executed successfully.',
    'ChargingSessionData': [
        {
            'stationID': '5:11649381',
            'stationName': 'PNNL / ESC 2',
            'portNumber': '2',
            'Address': '3340 Stevens Drive, Richland, Washington, 99354, United States',
            'City': 'Richland',
            'State': 'Washington',
            'Country': 'United States',
            'postalCode': '99354',
            'sessionID': 97610669,
            'Energy': 8.148128,
            'startTime': datetime.datetime(2024, 12, 2, 14, 57, 51, tzinfo=<isodate.tzinfo.Utc object at 0x7f0857263a90>),
            'endTime': datetime.datetime(2024, 12, 2, 16, 21, 24, tzinfo=<isodate.tzinfo.Utc object at 0x7f0857263a90>),
            'totalChargingDuration': '01:23:09',
            'totalSessionDuration': '01:23:36',
            'userID': '52689171',
            'startBatteryPercentage': 0.0,
            'stopBatteryPercentage': 0.